# Confidence vs Cytescore analysis
## Prep

In [ ]:
import json
import random
import os
from pathlib import Path
import pandas as pd
import scanpy as sc
from r2 import download_from_r2, fetch_uploaded_r2_keys

In [ ]:
r2_keys_set = fetch_uploaded_r2_keys()
r2_keys = list(r2_keys_set)
unique_prefixes = set()
for key in r2_keys:
    prefix = key.split("/")[0] if "/" in key else key
    if prefix not in unique_prefixes:
        unique_prefixes.add(prefix)
list(unique_prefixes)

In [ ]:
annotated_h5ad_keys = [key for key in r2_keys if key.startswith("cytetype_pipeline_20260522_175813")]

## Download and read `h5ad`
### Download from r2

In [ ]:
seed = 15
random.seed(seed)
i = random.randint(0, len(annotated_h5ad_keys) - 1)
ACCESSION = annotated_h5ad_keys[i].split("/")[-1].split("_")[0]
LOCAL_PATH = Path("../..") / "tmp" / f"{annotated_h5ad_keys[i].split('/')[-1]}"
download_from_r2(annotated_h5ad_keys[i], LOCAL_PATH)
print(f"File {i}: {ACCESSION}")

### Read downloaded file

In [ ]:
adata = sc.read(LOCAL_PATH)
adata.uns["cytetype_jobDetails"]["report_url"]

## Prepare CyteOnto baseline
### Import CyteOnto data

In [ ]:
df_raw = pd.read_csv(Path("../..") / f"output/cyteonto_pipeline/20260526_121155/results/{ACCESSION}_cyteonto.csv")

### Deduplicate CyteOnto `DataFrame`

In [ ]:
df_raw["pair_label"] = df_raw["algorithm_label"] + ":" + df_raw["author_label"]
df = df_raw.copy()
df.drop_duplicates(subset="pair_label", inplace=True)
print(len(df), len(df["cytescore_similarity"].unique()))
assert len(df) >= len(df["cytescore_similarity"].unique()), "Number of unique cytescore_similarity values does not match number unique STATE-CyteType label pairs"

## Analyze `anndata` object

In [ ]:
CLUSTER_KEY = "leiden_merged"

payload = adata.uns["cytetype_results"]["result"]
cytetype_result = json.loads(payload) if isinstance(payload, str) else payload

confidence_by_cluster = {
    str(cluster_id): entry["latest"]["review"]["confidence"]
    for cluster_id, entry in cytetype_result["raw_annotations"].items()
}

adata.obs["cytetype_confidence"] = (
    adata.obs[CLUSTER_KEY].astype(str).map(confidence_by_cluster)
)

pd.DataFrame(
    {"cluster_id": confidence_by_cluster.keys(), "confidence": confidence_by_cluster.values()}
).sort_values("cluster_id")

In [ ]:
adata.obs["pair_label"] = (
    adata.obs["cytetype_annotation_leiden_merged"].astype(str)
    + ":"
    + adata.obs["cell_type"].astype(str)
)

assert set(adata.obs["cytetype_annotation_leiden_merged"].unique()) == set(df["algorithm_label"].unique())

cytescore_by_pair = (
    df[["pair_label", "cytescore_similarity"]]
    .drop_duplicates("pair_label")
    .set_index("pair_label")["cytescore_similarity"]
)
adata.obs["cytescore_similarity"] = adata.obs["pair_label"].map(cytescore_by_pair)

### UMAP colored by cluster confidence

In [ ]:
from scanpy.plotting._utils import set_colors_for_categorical_obs

print(adata.uns["cytetype_jobDetails"]["report_url"])

confidence_palette = {"Low": "#d73027", "Moderate": "#fee08b", "High": "#1a9850"}
set_colors_for_categorical_obs(adata, "cytetype_confidence", confidence_palette)

sc.pl.umap(
    adata,
    color=[
        "cell_type",
        "cytetype_annotation_leiden_merged",
        "cytescore_similarity",
        "cytetype_confidence",
    ],
    title=[
        f"{ACCESSION}: STATE labels",
        f"{ACCESSION}: CyteType annotations",
        f"{ACCESSION}: CyteOnto cytescore similarity",
        f"{ACCESSION}: CyteType cluster confidence",
    ],
    ncols=2,
    legend_loc="right margin",
    size=30,
    wspace=0.4,
)

# Clean up `h5ad`

In [ ]:
adata.obs

In [ ]:
# os.remove(LOCAL_PATH)